# **My Music Revisits:** Audit of Data

In [1]:
import json
import re
import sys
import zipfile

from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

In [2]:
pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

In [3]:
root = Path.cwd()

for path in [Path.cwd(), *Path.cwd().parents]:
    if (path / "data").exists() and (path / "requirements.txt").exists():
        root = path
        break

sys.path.insert(0, str(root))

In [4]:
data_root = root / "data"
raw_root = data_root / "raw"
raw_csv = raw_root / "csv"
raw_json = raw_root / "json_zips"

interim_root = data_root / "interim"
interim_csv = interim_root / "csv"
interim_json = interim_root / "json"

processed_root = data_root / "processed"
parquet_root = processed_root / "parquet"

metadata_root = data_root / "metadata"

reports_root = root / "reports"
tables_root = reports_root / "tables" / "data_check"
tables_root.mkdir(parents=True, exist_ok=True)
metadata_root.mkdir(parents=True, exist_ok=True)

print("Project Root:", root)
print("Data Directory Exists:", data_root.exists())
print("Raw CSV Directory Exists:", raw_csv.exists())
print("Raw JSON Zip Directory Exists:", raw_json.exists())
print("Interim CSV Directory Exists:", interim_csv.exists())
print("Interim JSON Directory Exists:", interim_json.exists())
print("Processed Parquet Directory Exists:", parquet_root.exists())
print("Metadata Directory Exists:", metadata_root.exists())
print("Tables Directory:", tables_root)

Project Root: C:\Users\sumai\Documents\Coding\my-music-revisits
Data Directory Exists: True
Raw CSV Directory Exists: True
Raw JSON Zip Directory Exists: True
Interim CSV Directory Exists: False
Interim JSON Directory Exists: False
Processed Parquet Directory Exists: False
Metadata Directory Exists: True
Tables Directory: C:\Users\sumai\Documents\Coding\my-music-revisits\reports\tables\data_check


In [5]:
def clean_name(value: str) -> str:
    value = str(value).strip().lower()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    value = re.sub(r"_+", "_", value)
    return value.strip("_")

In [6]:
def table_name(path: Path) -> str:
    name = path.name
    name = name.replace(".json.zip", "")
    name = name.replace(".parquet", "")
    name = name.replace(".csv", "")
    name = name.replace(".json", "")
    return clean_name(name)

In [7]:
def kind(path: Path) -> str:
    if path.name.endswith(".json.zip"):
        return "json_zip"

    if path.name.endswith(".parquet"):
        return "parquet"

    if path.name.endswith(".csv"):
        return "csv"

    if path.name.endswith(".json"):
        return "json"

    return path.suffix.replace(".", "")


In [8]:
def layer(path: Path) -> str:
    path_text = path.as_posix()

    if "/data/raw/" in path_text:
        return "raw"

    if "/data/interim/" in path_text:
        return "interim"

    if "/data/processed/" in path_text:
        return "processed"

    if "/data/metadata/" in path_text:
        return "metadata"

    return "other"

In [9]:
def key(path: Path) -> str:
    return f"{layer(path)}__{kind(path)}__{table_name(path)}"

In [37]:
def save_table(dataframe: pd.DataFrame, name: str) -> Path:
    output = tables_root / name
    dataframe.to_csv(output, index=False)
    return output

In [38]:
def memory(dataframe: pd.DataFrame) -> float:
    return round(
        dataframe.memory_usage(deep=True).sum() / 1_000_000,
        3,
    )

In [39]:
def hashable(value: Any) -> Any:
    if isinstance(value, (list, dict)):
        return json.dumps(
            value,
            ensure_ascii=False,
            sort_keys=True,
        )

    return value

In [40]:
def duplicates(dataframe: pd.DataFrame) -> int:
    if dataframe.empty:
        return 0

    work = dataframe.map(hashable)
    return int(work.duplicated().sum())

In [41]:
def uniques(series: pd.Series) -> int:
    return int(
        series
        .dropna()
        .map(hashable)
        .nunique()
    )

In [42]:
def normalize_json(data: Any) -> pd.DataFrame:
    if isinstance(data, list):
        return pd.json_normalize(data)

    if isinstance(data, dict):
        for value in data.values():
            if isinstance(value, list):
                return pd.json_normalize(value)

        return pd.json_normalize(data)

    return pd.DataFrame()


def read_json_zip(path: Path) -> pd.DataFrame:
    with zipfile.ZipFile(path, "r") as zip_file:
        names = zip_file.namelist()

        if not names:
            return pd.DataFrame()

        with zip_file.open(names[0]) as file:
            data = json.load(file)

    return normalize_json(data)


def read_json_file(path: Path) -> pd.DataFrame:
    with open(path, "r", encoding="utf-8") as file:
        data = json.load(file)

    return normalize_json(data)


def read_table(path: Path) -> pd.DataFrame:
    if path.name.endswith(".parquet"):
        return pd.read_parquet(path)

    if path.name.endswith(".csv"):
        return pd.read_csv(path, low_memory=False)

    if path.name.endswith(".json.zip"):
        return read_json_zip(path)

    if path.name.endswith(".json"):
        return read_json_file(path)

    raise ValueError(f"Unsupported file type: {path.name}")

In [43]:
def family(column: str) -> str:
    name = clean_name(column)
    tokens = name.split("_")

    identifiers = [
        "identifier",
        "reference",
        "isrc",
        "upc",
        "playlist_id",
        "container_id",
        "recommendation_id",
        "session_id",
        "event_id",
        "track_id",
        "item_id",
        "adam_id",
        "catalog_id",
    ]

    dates = [
        "date",
        "time",
        "timestamp",
        "created",
        "modified",
        "started",
        "ended",
        "played",
        "received",
        "added",
    ]

    durations = [
        "duration",
        "millisecond",
        "milliseconds",
        "second",
        "seconds",
        "minute",
        "minutes",
        "position",
    ]

    counts = [
        "count",
        "plays",
        "skips",
        "rank",
        "ranking",
        "total",
        "number",
    ]

    metadata = [
        "song",
        "track",
        "title",
        "artist",
        "album",
        "genre",
        "disc",
        "movement",
        "work",
        "composer",
        "rating",
        "explicit",
        "artwork",
        "bpm",
        "tempo",
    ]

    contexts = [
        "source",
        "feature",
        "container",
        "playlist",
        "device",
        "os",
        "platform",
        "country",
        "language",
        "offline",
        "quality",
        "codec",
        "format",
        "shuffle",
        "repeat",
    ]

    preferences = [
        "favorite",
        "like",
        "dislike",
        "preference",
        "loved",
        "banned",
    ]

    if any(term in name for term in preferences):
        return "preference"

    if "id" in tokens or any(term in name for term in identifiers):
        return "identifier"

    if any(term in name for term in dates):
        return "date_time"

    if any(term in name for term in durations):
        return "duration"

    if any(term in name for term in counts):
        return "count"

    if any(term in name for term in contexts):
        return "context"

    if any(term in name for term in metadata):
        return "music_metadata"

    return "other"

In [44]:
def lookup(dataframe: pd.DataFrame) -> dict[str, str]:
    return {clean_name(column): column for column in dataframe.columns}

In [45]:
def find_column(dataframe: pd.DataFrame, candidates: list[str]) -> str | None:
    columns = lookup(dataframe)

    for candidate in candidates:
        clean_candidate = clean_name(candidate)

        if clean_candidate in columns:
            return columns[clean_candidate]

    return None

In [46]:
def join_value(value: Any) -> str | None:
    if pd.isna(value):
        return None

    text = str(value).strip()

    if not text:
        return None

    if re.fullmatch(r"\d+\.0", text):
        text = text[:-2]

    return text.lower()

In [47]:
def parse_dates(series: pd.Series) -> pd.Series:
    parsed = pd.to_datetime(series, errors="coerce", utc=True)

    if parsed.notna().mean() == 0:
        cleaned = (
            series
            .astype(str)
            .str.replace(r"\[UTC\]$", "", regex=True)
            .replace("nan", np.nan)
        )

        parsed = pd.to_datetime(cleaned, errors="coerce", utc=True)

    return parsed

In [49]:
def date_summary(series: pd.Series) -> dict[str, Any]:
    non_null = int(series.notna().sum())

    if non_null == 0:
        return {
            "parse_rate": 0,
            "min_date": pd.NaT,
            "max_date": pd.NaT,
        }

    parsed = parse_dates(series)
    parse_rate = round(parsed.notna().sum() / non_null, 4)

    return {
        "parse_rate": parse_rate,
        "min_date": parsed.min(),
        "max_date": parsed.max(),
    }

In [50]:
patterns = [
    "data/raw/csv/*.csv",
    "data/raw/json_zips/*.json.zip",
    "data/interim/csv/*.csv",
    "data/interim/json/*.json",
    "data/processed/parquet/*.parquet",
    "data/metadata/*.csv",
]

paths = []

for pattern in patterns:
    paths.extend(root.glob(pattern))

paths = sorted(set(paths))

In [51]:
files = pd.DataFrame(
    {
        "table_key": key(path),
        "file_name": path.name,
        "table_name": table_name(path),
        "file_type": kind(path),
        "data_layer": layer(path),
        "relative_path": str(path.relative_to(root)),
        "size_mb": round(path.stat().st_size / 1_000_000, 3),
    }
    for path in paths
)

files = files.sort_values(
    ["data_layer", "file_type", "table_name"]
).reset_index(drop=True)

save_table(files, "file_manifest.csv")

print("Discovered Files:", len(files))
display(files)

Discovered Files: 25


,table_key,file_name,table_name,file_type,data_layer,relative_path,size_mb
0,metadata__csv__json_conversion_report,json_conversion_report.csv,json_conversion_report,csv,metadata,data\metadata\json_conversion_report.csv,0.002
1,metadata__csv__processed_table_report,processed_table_report.csv,processed_table_report,csv,metadata,data\metadata\processed_table_report.csv,0.000
2,metadata__csv__raw_column_profile,raw_column_profile.csv,raw_column_profile,csv,metadata,data\metadata\raw_column_profile.csv,0.154
3,metadata__csv__raw_data_inventory,raw_data_inventory.csv,raw_data_inventory,csv,metadata,data\metadata\raw_data_inventory.csv,0.009
4,raw__csv__container_details,container_details.csv,container_details,csv,raw,data\raw\csv\container_details.csv,0.021
5,raw__csv__container_origin,container_origin.csv,container_origin,csv,raw,data\raw\csv\container_origin.csv,0.011
6,raw__csv__favorites,favorites.csv,favorites,csv,raw,data\raw\csv\favorites.csv,0.012
7,raw__csv__feature_statistics,feature_statistics.csv,feature_statistics,csv,raw,data\raw\csv\feature_statistics.csv,0.010
8,raw__csv__music_play_activity,music_play_activity.csv,music_play_activity,csv,raw,data\raw\csv\music_play_activity.csv,29.859
9,raw__csv__play_history_daily_tracks,play_history_daily_tracks.csv,play_history_daily_tracks,csv,raw,data\raw\csv\play_history_daily_tracks.csv,0.717


In [53]:
tables = {}
load_rows = []

for path in paths:
    table_key = key(path)

    try:
        dataframe = read_table(path)
        tables[table_key] = dataframe

        load_rows.append(
            {
                "table_key": table_key,
                "file_name": path.name,
                "table_name": table_name(path),
                "file_type": kind(path),
                "data_layer": layer(path),
                "relative_path": str(path.relative_to(root)),
                "load_status": "Loaded",
                "row_count": dataframe.shape[0],
                "column_count": dataframe.shape[1],
                "memory_mb": memory(dataframe),
                "duplicate_rows": duplicates(dataframe),
                "error_message": "",
            }
        )

    except Exception as error:
        load_rows.append(
            {
                "table_key": table_key,
                "file_name": path.name,
                "table_name": table_name(path),
                "file_type": kind(path),
                "data_layer": layer(path),
                "relative_path": str(path.relative_to(root)),
                "load_status": "Failed",
                "row_count": np.nan,
                "column_count": np.nan,
                "memory_mb": np.nan,
                "duplicate_rows": np.nan,
                "error_message": str(error),
            }
        )

In [54]:
inventory = pd.DataFrame(load_rows)

inventory = inventory.sort_values(
    ["load_status", "data_layer", "row_count"],
    ascending=[True, True, False],
).reset_index(drop=True)

loaded = (
    inventory
    .query("load_status == 'Loaded'")
    .copy()
    .sort_values("row_count", ascending=False)
    .reset_index(drop=True)
)

save_table(inventory, "table_inventory.csv")

print("Loaded Tables:", len(tables))
print("Failed Tables:", int((inventory["load_status"] == "Failed").sum()))
print("Total Loaded Rows:", int(loaded["row_count"].sum()))
print("Total Loaded Columns:", int(loaded["column_count"].sum()))
print("Largest Table:", loaded.iloc[0]["table_key"])
print("Widest Table:", loaded.sort_values("column_count", ascending=False).iloc[0]["table_key"])

display(loaded)

Loaded Tables: 25
Failed Tables: 0
Total Loaded Rows: 83668
Total Loaded Columns: 428
Largest Table: raw__csv__music_play_activity
Widest Table: raw__csv__music_play_activity


,table_key,file_name,table_name,file_type,data_layer,relative_path,load_status,row_count,column_count,memory_mb,duplicate_rows,error_message
0,raw__csv__music_play_activity,music_play_activity.csv,music_play_activity,csv,raw,data\raw\csv\music_play_activity.csv,Loaded,22872,144,102.393,0,
1,raw__json_zip__identifier_information,identifier_information.json.zip,identifier_information,json_zip,raw,data\raw\json_zips\identifier_information.json...,Loaded,15260,2,1.943,0,
2,raw__json_zip__library_tracks,library_tracks.json.zip,library_tracks,json_zip,raw,data\raw\json_zips\library_tracks.json.zip,Loaded,10738,50,19.303,0,
3,raw__json_zip__library_import_tracks,library_import_tracks.json.zip,library_import_tracks,json_zip,raw,data\raw\json_zips\library_import_tracks.json.zip,Loaded,9547,22,15.104,0,
4,raw__csv__track_play_history,track_play_history.csv,track_play_history,csv,raw,data\raw\csv\track_play_history.csv,Loaded,5981,3,0.545,0,
5,raw__json_zip__library_albums,library_albums.json.zip,library_albums,json_zip,raw,data\raw\json_zips\library_albums.json.zip,Loaded,5255,6,1.679,0,
6,raw__csv__play_history_daily_tracks,play_history_daily_tracks.csv,play_history_daily_tracks,csv,raw,data\raw\csv\play_history_daily_tracks.csv,Loaded,5042,13,2.178,0,
7,raw__json_zip__library_activity,library_activity.json.zip,library_activity,json_zip,raw,data\raw\json_zips\library_activity.json.zip,Loaded,3842,35,7.868,0,
8,raw__json_zip__library_artists,library_artists.json.zip,library_artists,json_zip,raw,data\raw\json_zips\library_artists.json.zip,Loaded,3109,6,0.961,0,
9,metadata__csv__raw_column_profile,raw_column_profile.csv,raw_column_profile,csv,metadata,data\metadata\raw_column_profile.csv,Loaded,401,10,0.253,0,


In [55]:
versions = (
    loaded
    .assign(canonical_name=lambda dataframe: dataframe["file_name"].map(lambda value: table_name(Path(value))))
    [
        [
            "canonical_name",
            "table_key",
            "file_name",
            "table_name",
            "file_type",
            "data_layer",
            "relative_path",
            "row_count",
            "column_count",
            "memory_mb",
            "duplicate_rows",
        ]
    ]
    .sort_values(["canonical_name", "data_layer", "file_type"])
    .reset_index(drop=True)
)

In [56]:
def get_key(canonical: str, data_layer: str | None = None, file_type: str | None = None) -> str | None:
    matches = versions.query("canonical_name == @canonical").copy()

    if data_layer is not None:
        matches = matches.query("data_layer == @data_layer").copy()

    if file_type is not None:
        matches = matches.query("file_type == @file_type").copy()

    if matches.empty:
        return None

    layer_rank = {
        "processed": 1,
        "interim": 2,
        "raw": 3,
        "metadata": 4,
    }

    type_rank = {
        "parquet": 1,
        "csv": 2,
        "json": 3,
        "json_zip": 4,
    }

    matches["layer_rank"] = matches["data_layer"].map(layer_rank).fillna(99)
    matches["type_rank"] = matches["file_type"].map(type_rank).fillna(99)

    return (
        matches
        .sort_values(["layer_rank", "type_rank", "row_count"], ascending=[True, True, False])
        .iloc[0]["table_key"]
    )

In [30]:
contract_rows = [
    {
        "canonical_name": "play_activity_events",
        "preferred_key": get_key("play_activity_events", "processed", "parquet"),
        "project_role": "Primary Detailed Playback Event Table",
        "grain": "One playback activity record per event",
        "next_use": "EDA Core",
        "dashboard_use": "Playback friction, completion, source context, device context, and sessionization",
        "model_use": "Implicit feedback, completion quality, skip-adjusted affinity, recency, and context features",
        "reason": "This is the cleaned event-level behavioral table and should replace raw music_play_activity for most analysis.",
    },
    {
        "canonical_name": "listening_events",
        "preferred_key": get_key("listening_events", "processed", "parquet"),
        "project_role": "Primary Summary Listening Table",
        "grain": "Track-level listening summary record",
        "next_use": "EDA Core",
        "dashboard_use": "Listening volume, repeat behavior, time trends, track-level summaries, and skip summaries",
        "model_use": "Track affinity, repeat behavior, skip rate, and long-term preference features",
        "reason": "This is the clean summary layer derived from daily track listening history.",
    },
    {
        "canonical_name": "library_tracks_clean",
        "preferred_key": get_key("library_tracks_clean", "processed", "parquet"),
        "project_role": "Primary Track Metadata Dimension",
        "grain": "One library track record per resolved library track",
        "next_use": "EDA Core",
        "dashboard_use": "Track, artist, album, genre, release year, duration, library state, and metadata enrichment",
        "model_use": "Candidate universe, content features, identity resolution, and metadata similarity",
        "reason": "This is the most analysis-ready track metadata table.",
    },
    {
        "canonical_name": "favorites_clean",
        "preferred_key": get_key("favorites_clean", "processed", "parquet"),
        "project_role": "Explicit Preference Table",
        "grain": "One explicit preference record per item",
        "next_use": "EDA Core",
        "dashboard_use": "Preference alignment, favorites coverage, and library affinity",
        "model_use": "Explicit preference boost and negative preference penalty",
        "reason": "Explicit preference is sparse but high-value for recommender readiness.",
    },
    {
        "canonical_name": "identifier_information",
        "preferred_key": get_key("identifier_information", "processed", "parquet"),
        "project_role": "Identifier Lookup Table",
        "grain": "One identifier-to-title lookup record",
        "next_use": "EDA Support",
        "dashboard_use": "Fallback label resolution for opaque identifiers",
        "model_use": "Fallback identity resolution",
        "reason": "This table helps reconcile identifiers that do not join cleanly through the track metadata table.",
    },
    {
        "canonical_name": "library_import_tracks",
        "preferred_key": get_key("library_import_tracks", "processed", "parquet"),
        "project_role": "Imported Track Metadata Table",
        "grain": "One imported track record per imported source track",
        "next_use": "EDA Support",
        "dashboard_use": "Imported catalog coverage and secondary metadata enrichment",
        "model_use": "External identifiers, imported track candidates, and metadata matching if reliable",
        "reason": "This table may improve identity coverage but should not override library_tracks_clean without join validation.",
    },
    {
        "canonical_name": "library_playlists",
        "preferred_key": get_key("library_playlists", "processed", "parquet"),
        "project_role": "Playlist Metadata Table",
        "grain": "One library playlist or container record",
        "next_use": "EDA Support",
        "dashboard_use": "Playlist structure, playlist size, playlist recency, and curation context",
        "model_use": "Playlist membership and container-based affinity if joins are reliable",
        "reason": "This supports playlist and container analysis but should not be treated as listening behavior.",
    },
    {
        "canonical_name": "library_activity",
        "preferred_key": get_key("library_activity", "processed", "parquet"),
        "project_role": "Library Curation Activity Table",
        "grain": "One library or playlist activity transaction",
        "next_use": "EDA Support",
        "dashboard_use": "Library curation, playlist edits, additions, and modification timelines",
        "model_use": "Recency and curation signals if transaction types are reliable",
        "reason": "This table is structurally sparse and should be analyzed by transaction type in EDA.",
    },
    {
        "canonical_name": "recent_impressions",
        "preferred_key": get_key("recent_impressions", "raw", "csv"),
        "project_role": "Recommendation Impression Sample",
        "grain": "One recent impression record",
        "next_use": "EDA Support",
        "dashboard_use": "Recommendation-adjacent exposure sample and validation only",
        "model_use": "Weak exposure signal only",
        "reason": "This table is small and duplicated, so it should not be used as a complete recommendation evaluation log.",
    },
    {
        "canonical_name": "recently_played_tracks",
        "preferred_key": get_key("recently_played_tracks", "raw", "csv"),
        "project_role": "Recent Track Snapshot",
        "grain": "One recent track summary record",
        "next_use": "EDA Support",
        "dashboard_use": "Recent behavior validation",
        "model_use": "Short-term recency checks",
        "reason": "This table is a recent snapshot, not the main behavioral history.",
    },
    {
        "canonical_name": "recently_played_containers",
        "preferred_key": get_key("recently_played_containers", "raw", "csv"),
        "project_role": "Recent Container Snapshot",
        "grain": "One recent container summary record",
        "next_use": "EDA Support",
        "dashboard_use": "Recent source and container validation",
        "model_use": "Recent container context if identifiers resolve",
        "reason": "This table is useful for recent context validation, not complete history.",
    },
    {
        "canonical_name": "container_details",
        "preferred_key": get_key("container_details", "raw", "csv"),
        "project_role": "Container Context Table",
        "grain": "One container detail record",
        "next_use": "EDA Support",
        "dashboard_use": "Container-level listening summaries and source context",
        "model_use": "Container affinity if joins resolve",
        "reason": "This may enrich playlist and source analysis but should be joined only after identifier validation.",
    },
    {
        "canonical_name": "container_origin",
        "preferred_key": get_key("container_origin", "raw", "csv"),
        "project_role": "Container Origin Table",
        "grain": "One container origin summary record",
        "next_use": "EDA Support",
        "dashboard_use": "Container origin and source attribution",
        "model_use": "Context features if joins resolve",
        "reason": "This table supports source attribution but is not a behavioral source of truth.",
    },
    {
        "canonical_name": "top_content",
        "preferred_key": get_key("top_content", "raw", "csv"),
        "project_role": "Validation Summary Table",
        "grain": "One exported top-content summary record",
        "next_use": "Validation Only",
        "dashboard_use": "Aggregate validation only",
        "model_use": "Do not use directly",
        "reason": "This is useful for checking computed metrics but should not drive the dashboard.",
    },
    {
        "canonical_name": "play_statistics",
        "preferred_key": get_key("play_statistics", "raw", "csv"),
        "project_role": "Validation Summary Table",
        "grain": "One exported play-statistic summary record",
        "next_use": "Validation Only",
        "dashboard_use": "Aggregate validation only",
        "model_use": "Do not use directly",
        "reason": "This is a validation table, not an event table.",
    },
    {
        "canonical_name": "feature_statistics",
        "preferred_key": get_key("feature_statistics", "raw", "csv"),
        "project_role": "Validation Summary Table",
        "grain": "One exported feature-statistic summary record",
        "next_use": "Validation Only",
        "dashboard_use": "Aggregate validation only",
        "model_use": "Do not use directly",
        "reason": "This is useful for validating source and feature totals.",
    },
    {
        "canonical_name": "music_play_activity",
        "preferred_key": get_key("music_play_activity", "raw", "csv"),
        "project_role": "Raw Detailed Playback Event Source",
        "grain": "One raw playback activity record per event",
        "next_use": "Raw Fallback Only",
        "dashboard_use": "Only when a needed field is missing from play_activity_events",
        "model_use": "Only when a needed field is missing from play_activity_events",
        "reason": "This table is wide and sparse. It should be preserved for lineage but not used as the default analytical table.",
    },
    {
        "canonical_name": "play_history_daily_tracks",
        "preferred_key": get_key("play_history_daily_tracks", "raw", "csv"),
        "project_role": "Raw Summary Listening Source",
        "grain": "One raw daily-track summary record",
        "next_use": "Raw Fallback Only",
        "dashboard_use": "Only when a needed field is missing from listening_events",
        "model_use": "Only when a needed field is missing from listening_events",
        "reason": "This is the raw source behind the cleaner listening_events table.",
    },
]

In [57]:
raw_contracts = pd.DataFrame(contract_rows)

missing_contracts = (
    raw_contracts
    .query("preferred_key.isna()", engine="python")
    .copy()
    .reset_index(drop=True)
)

save_table(missing_contracts, "missing_contracts.csv")

print("Missing Expected Contract Tables:", len(missing_contracts))
display(missing_contracts)

Missing Expected Contract Tables: 8


,canonical_name,preferred_key,project_role,grain,next_use,dashboard_use,model_use,reason
0,play_activity_events,None,Primary Detailed Playback Event Table,One playback activity record per event,EDA Core,"Playback friction, completion, source context,...","Implicit feedback, completion quality, skip-ad...",This is the cleaned event-level behavioral tab...
1,listening_events,None,Primary Summary Listening Table,Track-level listening summary record,EDA Core,"Listening volume, repeat behavior, time trends...","Track affinity, repeat behavior, skip rate, an...",This is the clean summary layer derived from d...
2,library_tracks_clean,None,Primary Track Metadata Dimension,One library track record per resolved library ...,EDA Core,"Track, artist, album, genre, release year, dur...","Candidate universe, content features, identity...",This is the most analysis-ready track metadata...
3,favorites_clean,None,Explicit Preference Table,One explicit preference record per item,EDA Core,"Preference alignment, favorites coverage, and ...",Explicit preference boost and negative prefere...,Explicit preference is sparse but high-value f...
4,identifier_information,None,Identifier Lookup Table,One identifier-to-title lookup record,EDA Support,Fallback label resolution for opaque identifiers,Fallback identity resolution,This table helps reconcile identifiers that do...
5,library_import_tracks,None,Imported Track Metadata Table,One imported track record per imported source ...,EDA Support,Imported catalog coverage and secondary metada...,"External identifiers, imported track candidate...",This table may improve identity coverage but s...
6,library_playlists,None,Playlist Metadata Table,One library playlist or container record,EDA Support,"Playlist structure, playlist size, playlist re...",Playlist membership and container-based affini...,This supports playlist and container analysis ...
7,library_activity,None,Library Curation Activity Table,One library or playlist activity transaction,EDA Support,"Library curation, playlist edits, additions, a...",Recency and curation signals if transaction ty...,This table is structurally sparse and should b...


In [58]:
contracts = raw_contracts.dropna(subset=["preferred_key"]).reset_index(drop=True)

contracts = contracts.merge(
    loaded[
        [
            "table_key",
            "file_name",
            "data_layer",
            "file_type",
            "relative_path",
            "row_count",
            "column_count",
            "memory_mb",
            "duplicate_rows",
        ]
    ],
    left_on="preferred_key",
    right_on="table_key",
    how="left",
)

contracts = contracts.drop(columns=["table_key"])

eda_tables = (
    contracts
    .query("next_use in ['EDA Core', 'EDA Support', 'Validation Only']")
    .copy()
    .sort_values(["next_use", "row_count"], ascending=[True, False])
    .reset_index(drop=True)
)

chosen_keys = contracts["preferred_key"].dropna().drop_duplicates().tolist()
eda_keys = eda_tables["preferred_key"].dropna().drop_duplicates().tolist()

save_table(contracts, "table_contracts.csv")
save_table(eda_tables, "eda_input_tables.csv")

print("Selected Tables for Detailed Validation:", len(chosen_keys))
print("EDA Input Tables:", len(eda_keys))

display(contracts)
display(eda_tables)

Selected Tables for Detailed Validation: 10
EDA Input Tables: 8


,canonical_name,preferred_key,project_role,grain,next_use,dashboard_use,model_use,reason,file_name,data_layer,file_type,relative_path,row_count,column_count,memory_mb,duplicate_rows
0,recent_impressions,raw__csv__recent_impressions,Recommendation Impression Sample,One recent impression record,EDA Support,Recommendation-adjacent exposure sample and va...,Weak exposure signal only,"This table is small and duplicated, so it shou...",recent_impressions.csv,raw,csv,data\raw\csv\recent_impressions.csv,359,5,0.117,108
1,recently_played_tracks,raw__csv__recently_played_tracks,Recent Track Snapshot,One recent track summary record,EDA Support,Recent behavior validation,Short-term recency checks,"This table is a recent snapshot, not the main ...",recently_played_tracks.csv,raw,csv,data\raw\csv\recently_played_tracks.csv,200,22,0.192,0
2,recently_played_containers,raw__csv__recently_played_containers,Recent Container Snapshot,One recent container summary record,EDA Support,Recent source and container validation,Recent container context if identifiers resolve,This table is useful for recent context valida...,recently_played_containers.csv,raw,csv,data\raw\csv\recently_played_containers.csv,50,11,0.021,0
3,container_details,raw__csv__container_details,Container Context Table,One container detail record,EDA Support,Container-level listening summaries and source...,Container affinity if joins resolve,This may enrich playlist and source analysis b...,container_details.csv,raw,csv,data\raw\csv\container_details.csv,162,10,0.059,0
4,container_origin,raw__csv__container_origin,Container Origin Table,One container origin summary record,EDA Support,Container origin and source attribution,Context features if joins resolve,This table supports source attribution but is ...,container_origin.csv,raw,csv,data\raw\csv\container_origin.csv,69,8,0.021,0
5,top_content,raw__csv__top_content,Validation Summary Table,One exported top-content summary record,Validation Only,Aggregate validation only,Do not use directly,This is useful for checking computed metrics b...,top_content.csv,raw,csv,data\raw\csv\top_content.csv,196,8,0.044,0
6,play_statistics,raw__csv__play_statistics,Validation Summary Table,One exported play-statistic summary record,Validation Only,Aggregate validation only,Do not use directly,"This is a validation table, not an event table.",play_statistics.csv,raw,csv,data\raw\csv\play_statistics.csv,36,5,0.003,0
7,feature_statistics,raw__csv__feature_statistics,Validation Summary Table,One exported feature-statistic summary record,Validation Only,Aggregate validation only,Do not use directly,This is useful for validating source and featu...,feature_statistics.csv,raw,csv,data\raw\csv\feature_statistics.csv,126,6,0.020,0
8,music_play_activity,raw__csv__music_play_activity,Raw Detailed Playback Event Source,One raw playback activity record per event,Raw Fallback Only,Only when a needed field is missing from play_...,Only when a needed field is missing from play_...,This table is wide and sparse. It should be pr...,music_play_activity.csv,raw,csv,data\raw\csv\music_play_activity.csv,22872,144,102.393,0
9,play_history_daily_tracks,raw__csv__play_history_daily_tracks,Raw Summary Listening Source,One raw daily-track summary record,Raw Fallback Only,Only when a needed field is missing from liste...,Only when a needed field is missing from liste...,This is the raw source behind the cleaner list...,play_history_daily_tracks.csv,raw,csv,data\raw\csv\play_history_daily_tracks.csv,5042,13,2.178,0


,canonical_name,preferred_key,project_role,grain,next_use,dashboard_use,model_use,reason,file_name,data_layer,file_type,relative_path,row_count,column_count,memory_mb,duplicate_rows
0,recent_impressions,raw__csv__recent_impressions,Recommendation Impression Sample,One recent impression record,EDA Support,Recommendation-adjacent exposure sample and va...,Weak exposure signal only,"This table is small and duplicated, so it shou...",recent_impressions.csv,raw,csv,data\raw\csv\recent_impressions.csv,359,5,0.117,108
1,recently_played_tracks,raw__csv__recently_played_tracks,Recent Track Snapshot,One recent track summary record,EDA Support,Recent behavior validation,Short-term recency checks,"This table is a recent snapshot, not the main ...",recently_played_tracks.csv,raw,csv,data\raw\csv\recently_played_tracks.csv,200,22,0.192,0
2,container_details,raw__csv__container_details,Container Context Table,One container detail record,EDA Support,Container-level listening summaries and source...,Container affinity if joins resolve,This may enrich playlist and source analysis b...,container_details.csv,raw,csv,data\raw\csv\container_details.csv,162,10,0.059,0
3,container_origin,raw__csv__container_origin,Container Origin Table,One container origin summary record,EDA Support,Container origin and source attribution,Context features if joins resolve,This table supports source attribution but is ...,container_origin.csv,raw,csv,data\raw\csv\container_origin.csv,69,8,0.021,0
4,recently_played_containers,raw__csv__recently_played_containers,Recent Container Snapshot,One recent container summary record,EDA Support,Recent source and container validation,Recent container context if identifiers resolve,This table is useful for recent context valida...,recently_played_containers.csv,raw,csv,data\raw\csv\recently_played_containers.csv,50,11,0.021,0
5,top_content,raw__csv__top_content,Validation Summary Table,One exported top-content summary record,Validation Only,Aggregate validation only,Do not use directly,This is useful for checking computed metrics b...,top_content.csv,raw,csv,data\raw\csv\top_content.csv,196,8,0.044,0
6,feature_statistics,raw__csv__feature_statistics,Validation Summary Table,One exported feature-statistic summary record,Validation Only,Aggregate validation only,Do not use directly,This is useful for validating source and featu...,feature_statistics.csv,raw,csv,data\raw\csv\feature_statistics.csv,126,6,0.020,0
7,play_statistics,raw__csv__play_statistics,Validation Summary Table,One exported play-statistic summary record,Validation Only,Aggregate validation only,Do not use directly,"This is a validation table, not an event table.",play_statistics.csv,raw,csv,data\raw\csv\play_statistics.csv,36,5,0.003,0


In [61]:
column_rows = []

for table_key in chosen_keys:
    dataframe = tables[table_key]
    record = loaded.loc[loaded["table_key"] == table_key].iloc[0]

    for column in dataframe.columns:
        series = dataframe[column]
        row_count = dataframe.shape[0]
        missing_count = int(series.isna().sum())
        non_null_count = int(series.notna().sum())

        column_rows.append(
            {
                "table_key": table_key,
                "table_name": record["table_name"],
                "data_layer": record["data_layer"],
                "column_name": column,
                "column_family": family(column),
                "dtype": str(series.dtype),
                "row_count": row_count,
                "non_null_count": non_null_count,
                "missing_count": missing_count,
                "missing_rate": (
                    round(
                        missing_count / row_count,
                        4,
                    )
                    if row_count
                    else 0
                ),
                "unique_count": uniques(series),

            }
        )

In [62]:
columns = pd.DataFrame(column_rows)

columns = columns.sort_values(
    ["table_key", "column_family", "missing_rate", "column_name"],
    ascending=[True, True, False, True],
).reset_index(drop=True)

save_table(columns, "column_inventory_selected_tables.csv")

print("Profiled Columns:", len(columns))
display(columns)

Profiled Columns: 232


,table_key,table_name,data_layer,column_name,column_family,dtype,row_count,non_null_count,missing_count,missing_rate,unique_count
0,raw__csv__container_details,container_details,raw,Container Description,context,object,162,103,59,0.364,74
1,raw__csv__container_details,container_details,raw,Container Type,context,object,162,162,0,0.000,5
2,raw__csv__container_details,container_details,raw,Play Count,count,int64,162,162,0,0.000,21
3,raw__csv__container_details,container_details,raw,Date Created,date_time,int64,162,162,0,0.000,35
4,raw__csv__container_details,container_details,raw,Last Played,date_time,int64,162,162,0,0.000,162
...,...,...,...,...,...,...,...,...,...,...,...
227,raw__csv__top_content,top_content,raw,First Played,date_time,int64,196,196,0,0.000,172
228,raw__csv__top_content,top_content,raw,Last Played,date_time,int64,196,196,0,0.000,173
229,raw__csv__top_content,top_content,raw,Last Received,date_time,int64,196,196,0,0.000,144
230,raw__csv__top_content,top_content,raw,Play Duration Milliseconds,duration,int64,196,196,0,0.000,191


In [64]:
missing = (
    columns
    .groupby(
        [
            "table_key",
            "table_name",
            "data_layer",
        ],
        as_index=False,
    )
    .agg(
        column_count=("column_name", "count"),
        average_missing_rate=("missing_rate", "mean"),
        max_missing_rate=("missing_rate", "max"),
        fully_missing_columns=(
            "missing_rate",
            lambda values: int((values == 1).sum()),
        ),
        mostly_missing_columns=(
            "missing_rate",
            lambda values: int(
                (values >= 0.75).sum()
            ),
        ),
    )
)

missing["average_missing_rate"] = (
    missing["average_missing_rate"].round(4)
)

missing["max_missing_rate"] = (
    missing["max_missing_rate"].round(4)
)

missing = missing.sort_values(
    [
        "average_missing_rate",
        "mostly_missing_columns",
    ],
    ascending=[False, False],
).reset_index(drop=True)

save_table(
    missing,
    "table_missingness.csv",
)

display(missing)

,table_key,table_name,data_layer,column_count,average_missing_rate,max_missing_rate,fully_missing_columns,mostly_missing_columns
0,raw__csv__music_play_activity,music_play_activity,raw,144,0.546,1.000,32,64
1,raw__csv__recently_played_containers,recently_played_containers,raw,11,0.226,1.000,2,2
2,raw__csv__recent_impressions,recent_impressions,raw,5,0.164,0.549,0,0
3,raw__csv__container_details,container_details,raw,10,0.138,1.000,1,1
4,raw__csv__container_origin,container_origin,raw,8,0.129,1.000,1,1
5,raw__csv__recently_played_tracks,recently_played_tracks,raw,22,0.019,0.220,0,0
6,raw__csv__play_history_daily_tracks,play_history_daily_tracks,raw,13,0.006,0.077,0,0
7,raw__csv__feature_statistics,feature_statistics,raw,6,0.000,0.000,0,0
8,raw__csv__play_statistics,play_statistics,raw,5,0.000,0.000,0,0
9,raw__csv__top_content,top_content,raw,8,0.000,0.000,0,0


In [65]:
families = (
    columns
    .groupby(["table_key", "column_family"], as_index=False)
    .agg(column_count=("column_name", "count"))
    .pivot(index="table_key", columns="column_family", values="column_count")
    .fillna(0)
    .astype(int)
    .reset_index()
)

save_table(families, "column_family_summary_selected_tables.csv")

display(families)

column_family,table_key,context,count,date_time,duration,identifier,music_metadata,other
0,raw__csv__container_details,2,1,2,1,0,3,1
1,raw__csv__container_origin,0,1,2,1,0,3,1
2,raw__csv__feature_statistics,2,0,3,1,0,0,0
3,raw__csv__music_play_activity,29,4,8,9,30,5,59
4,raw__csv__play_history_daily_tracks,1,3,1,1,2,1,4
5,raw__csv__play_statistics,1,0,3,1,0,0,0
6,raw__csv__recent_impressions,1,1,1,0,1,0,1
7,raw__csv__recently_played_containers,2,3,2,0,2,1,1
8,raw__csv__recently_played_tracks,3,3,5,5,2,1,3
9,raw__csv__top_content,1,2,3,1,0,0,1


In [68]:
date_rows = []

date_columns = (
    columns
    .query("column_family == 'date_time'")
    .copy()
    .sort_values(
        ["table_key", "column_name"]
    )
)

for _, row in date_columns.iterrows():
    table_key = row["table_key"]
    column = row["column_name"]

    dataframe = tables[table_key]
    parsed = date_summary(dataframe[column])

    date_rows.append(
        {
            "table_key": table_key,
            "table_name": row["table_name"],
            "column_name": column,
            "dtype": row["dtype"],
            "non_null_count": row["non_null_count"],
            "unique_count": row["unique_count"],
            "parse_rate": parsed["parse_rate"],
            "min_date": parsed["min_date"],
            "max_date": parsed["max_date"],
        }
    )

dates = pd.DataFrame(date_rows)

if not dates.empty:
    dates = dates.sort_values(
        [
            "parse_rate",
            "table_key",
            "column_name",
        ],
        ascending=[False, True, True],
    ).reset_index(drop=True)

save_table(
    dates,
    "date_audit.csv",
)

display(dates)

C:\Users\sumai\AppData\Local\Temp\ipykernel_14288\3649152367.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(cleaned, errors="coerce", utc=True)
C:\Users\sumai\AppData\Local\Temp\ipykernel_14288\3649152367.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series, errors="coerce", utc=True)
C:\Users\sumai\AppData\Local\Temp\ipykernel_14288\3649152367.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(cleaned, errors="coerce", utc=True)
C:\Users\sumai\AppData\Local\Temp\ipykernel_14288\3649152367

,table_key,table_name,column_name,dtype,non_null_count,unique_count,parse_rate,min_date,max_date
0,raw__csv__container_details,container_details,Date Created,int64,162,35,1.000,1970-01-01 00:00:00.020251018+00:00,1970-01-01 00:00:00.020260311+00:00
1,raw__csv__container_details,container_details,Last Played,int64,162,162,1.000,1970-01-01 00:29:20.766207746+00:00,1970-01-01 00:29:33.191684120+00:00
2,raw__csv__container_origin,container_origin,Date Created,int64,69,35,1.000,1970-01-01 00:00:00.020251018+00:00,1970-01-01 00:00:00.020260311+00:00
3,raw__csv__container_origin,container_origin,Last Played,int64,69,69,1.000,1970-01-01 00:29:20.766207746+00:00,1970-01-01 00:29:33.191684120+00:00
4,raw__csv__feature_statistics,feature_statistics,Date Created,int64,126,36,1.000,1970-01-01 00:00:00.020251018+00:00,1970-01-01 00:00:00.020260311+00:00
5,raw__csv__feature_statistics,feature_statistics,First Played,int64,126,126,1.000,1970-01-01 00:29:20.766123819+00:00,1970-01-01 00:29:33.189596385+00:00
6,raw__csv__feature_statistics,feature_statistics,Last Played,int64,126,126,1.000,1970-01-01 00:29:20.766207746+00:00,1970-01-01 00:29:33.191684120+00:00
7,raw__csv__music_play_activity,music_play_activity,Event Post Date Time,int64,22872,1731,1.000,1970-01-01 00:00:00.029345833+00:00,1970-01-01 00:00:00.029558690+00:00
8,raw__csv__play_history_daily_tracks,play_history_daily_tracks,Date Played,int64,5042,133,1.000,1970-01-01 00:00:00.020251018+00:00,1970-01-01 00:00:00.020260310+00:00
9,raw__csv__play_statistics,play_statistics,Date Created,int64,36,36,1.000,1970-01-01 00:00:00.020251018+00:00,1970-01-01 00:00:00.020260311+00:00


In [69]:
numeric_rows = []

numeric_columns = (
    columns
    .query(
        "column_family in ['duration', 'count']"
    )
    .copy()
    .sort_values(
        ["table_key", "column_name"]
    )
)

for _, row in numeric_columns.iterrows():
    table_key = row["table_key"]
    column = row["column_name"]

    values = pd.to_numeric(
        tables[table_key][column],
        errors="coerce",
    )

    if values.notna().sum() == 0:
        continue

    numeric_rows.append(
        {
            "table_key": table_key,
            "table_name": row["table_name"],
            "column_name": column,
            "column_family": row["column_family"],
            "non_null_numeric_count": int(
                values.notna().sum()
            ),
            "minimum": values.min(),
            "median": values.median(),
            "mean": values.mean(),
            "maximum": values.max(),
            "zero_count": int(
                (values == 0).sum()
            ),
            "negative_count": int(
                (values < 0).sum()
            ),
        }
    )

numeric = pd.DataFrame(numeric_rows)

save_table(
    numeric,
    "numeric_audit.csv",
)

display(numeric)

,table_key,table_name,column_name,column_family,non_null_numeric_count,minimum,median,mean,maximum,zero_count,negative_count
0,raw__csv__container_details,container_details,Play Count,count,162,1.000,2.000,5.531,140.000,0,0
1,raw__csv__container_details,container_details,Play Duration Milliseconds,duration,162,"30,863.000","332,207.000","926,311.611","30,178,797.000",0,0
2,raw__csv__container_origin,container_origin,Play Count,count,69,1.000,5.000,12.986,161.000,0,0
3,raw__csv__container_origin,container_origin,Play Duration Milliseconds,duration,69,"37,537.000","755,212.000","2,174,818.565","34,123,795.000",0,0
4,raw__csv__feature_statistics,feature_statistics,Play Duration Milliseconds,duration,126,"30,000.000","463,135.500","1,303,678.587","34,311,391.000",0,0
5,raw__csv__music_play_activity,music_play_activity,Display Count,count,1068,0.000,0.000,0.000,0.000,1068,0
6,raw__csv__music_play_activity,music_play_activity,End Position In Milliseconds,duration,13820,0.000,"9,425.500","60,836.995","6,105,000.000",743,0
7,raw__csv__music_play_activity,music_play_activity,Media Duration In Milliseconds,duration,22870,0.000,"194,733.000","203,240.891","7,200,500.000",20,0
8,raw__csv__music_play_activity,music_play_activity,Milliseconds Since Play,duration,22872,0.000,200.000,"1,432,787.536","1,121,113,000.000",595,0
9,raw__csv__music_play_activity,music_play_activity,Play Duration Milliseconds,duration,13820,0.000,"5,006.500","49,777.413","684,632.000",340,0


In [70]:
millisecond_rows = []

for _, row in numeric_columns.iterrows():
    table_key = row["table_key"]
    column = row["column_name"]
    column_clean = clean_name(column)

    if (
        "millisecond" not in column_clean
        and "millis" not in column_clean
    ):
        continue

    values = pd.to_numeric(
        tables[table_key][column],
        errors="coerce",
    )

    if values.notna().sum() == 0:
        continue

    minutes = values / 60_000

    millisecond_rows.append(
        {
            "table_key": table_key,
            "table_name": row["table_name"],
            "column_name": column,
            "non_null_count": int(
                values.notna().sum()
            ),
            "total_hours": round(
                minutes.sum() / 60,
                3,
            ),
            "median_minutes": round(
                minutes.median(),
                3,
            ),
            "max_minutes": round(
                minutes.max(),
                3,
            ),
        }
    )

durations = pd.DataFrame(
    millisecond_rows
)

if not durations.empty:
    durations = durations.sort_values(
        "total_hours",
        ascending=False,
    ).reset_index(drop=True)

save_table(
    durations,
    "millisecond_audit.csv",
)

display(durations)

,table_key,table_name,column_name,non_null_count,total_hours,median_minutes,max_minutes
0,raw__csv__music_play_activity,music_play_activity,Milliseconds Since Play,22872,"9,102.977",0.003,"18,685.217"
1,raw__csv__music_play_activity,music_play_activity,Media Duration In Milliseconds,22870,"1,291.144",3.246,120.008
2,raw__csv__music_play_activity,music_play_activity,End Position In Milliseconds,13820,233.546,0.157,101.750
3,raw__csv__music_play_activity,music_play_activity,Play Duration Milliseconds,13820,191.090,0.083,11.411
4,raw__csv__play_history_daily_tracks,play_history_daily_tracks,Play Duration Milliseconds,5042,150.182,0.000,133.973
5,raw__csv__top_content,top_content,Play Duration Milliseconds,196,91.229,4.426,"1,258.315"
6,raw__csv__feature_statistics,feature_statistics,Play Duration Milliseconds,126,45.629,7.719,571.857
7,raw__csv__play_statistics,play_statistics,Play Duration Milliseconds,36,45.629,53.136,572.390
8,raw__csv__container_origin,container_origin,Play Duration Milliseconds,69,41.684,12.587,568.730
9,raw__csv__container_details,container_details,Play Duration Milliseconds,162,41.684,5.537,502.980


In [71]:
def unique_values(dataframe: pd.DataFrame, join_columns: list[str]) -> pd.Series:
    if len(join_columns) == 1:
        values = dataframe[join_columns[0]].map(join_value)
        return values.dropna().drop_duplicates()

    values = dataframe[join_columns].copy()
    values = values.map(join_value).dropna()

    if values.empty:
        return pd.Series(dtype="object")

    combined = values.astype(str).agg(" || ".join, axis=1)
    return combined.dropna().drop_duplicates()

In [72]:
def test_join(
    left_key: str,
    right_key: str,
    left_candidates: list[str],
    right_candidates: list[str],
    join_name: str,
) -> dict[str, Any]:
    left = tables[left_key]
    right = tables[right_key]

    left_columns = []

    for candidate in left_candidates:
        column = find_column(left, [candidate])

        if column is not None:
            left_columns.append(column)

    right_columns = []

    for candidate in right_candidates:
        column = find_column(right, [candidate])

        if column is not None:
            right_columns.append(column)

    if len(left_columns) != len(left_candidates) or len(right_columns) != len(right_candidates):
        return {
            "join_name": join_name,
            "left_table": left_key,
            "right_table": right_key,
            "left_columns": " | ".join(left_candidates),
            "right_columns": " | ".join(right_candidates),
            "left_unique_values": np.nan,
            "right_unique_values": np.nan,
            "overlap_unique_values": np.nan,
            "left_overlap_rate": np.nan,
            "right_overlap_rate": np.nan,
            "status": "Skipped Missing Column",
        }

    left_values = unique_values(left, left_columns)
    right_values = unique_values(right, right_columns)

    if len(left_values) == 0 or len(right_values) == 0:
        return {
            "join_name": join_name,
            "left_table": left_key,
            "right_table": right_key,
            "left_columns": " | ".join(left_columns),
            "right_columns": " | ".join(right_columns),
            "left_unique_values": len(left_values),
            "right_unique_values": len(right_values),
            "overlap_unique_values": 0,
            "left_overlap_rate": 0,
            "right_overlap_rate": 0,
            "status": "No Joinable Values",
        }

    left_set = set(left_values)
    right_set = set(right_values)
    overlap_count = len(left_set.intersection(right_set))

    return {
        "join_name": join_name,
        "left_table": left_key,
        "right_table": right_key,
        "left_columns": " | ".join(left_columns),
        "right_columns": " | ".join(right_columns),
        "left_unique_values": len(left_set),
        "right_unique_values": len(right_set),
        "overlap_unique_values": overlap_count,
        "left_overlap_rate": round(overlap_count / len(left_set), 4),
        "right_overlap_rate": round(overlap_count / len(right_set), 4),
        "status": "Tested",
    }

In [73]:
play_activity_key = get_key("play_activity_events", "processed", "parquet")
listening_key = get_key("listening_events", "processed", "parquet")
library_key = get_key("library_tracks_clean", "processed", "parquet")
favorites_key = get_key("favorites_clean", "processed", "parquet")
identifier_key = get_key("identifier_information", "processed", "parquet")
imports_key = get_key("library_import_tracks", "processed", "parquet")
playlists_key = get_key("library_playlists", "processed", "parquet")
container_key = get_key("container_details", "raw", "csv")
origin_key = get_key("container_origin", "raw", "csv")

join_specs = []

In [74]:
if listening_key and library_key:
    join_specs.extend(
        [
            {
                "join_name": "listening_to_library_by_reference",
                "left_key": listening_key,
                "right_key": library_key,
                "left_candidates": ["track_reference"],
                "right_candidates": ["apple_music_track_identifier"],
            },
            {
                "join_name": "listening_to_library_by_identifier",
                "left_key": listening_key,
                "right_key": library_key,
                "left_candidates": ["track_identifier"],
                "right_candidates": ["track_identifier"],
            },
            {
                "join_name": "listening_to_library_by_title_artist",
                "left_key": listening_key,
                "right_key": library_key,
                "left_candidates": ["title_from_description", "artist_from_description"],
                "right_candidates": ["title", "artist"],
            },
        ]
    )

In [75]:
if play_activity_key and library_key:
    join_specs.extend(
        [
            {
                "join_name": "activity_to_library_by_song",
                "left_key": play_activity_key,
                "right_key": library_key,
                "left_candidates": ["song_name"],
                "right_candidates": ["title"],
            },
            {
                "join_name": "activity_to_library_by_song_artist",
                "left_key": play_activity_key,
                "right_key": library_key,
                "left_candidates": ["song_name", "artist_name"],
                "right_candidates": ["title", "artist"],
            },
        ]
    )

In [76]:
if favorites_key and library_key:
    join_specs.extend(
        [
            {
                "join_name": "favorites_to_library_by_catalog_id",
                "left_key": favorites_key,
                "right_key": library_key,
                "left_candidates": ["item_reference"],
                "right_candidates": ["apple_music_track_identifier"],
            },
            {
                "join_name": "favorites_to_library_by_track_id",
                "left_key": favorites_key,
                "right_key": library_key,
                "left_candidates": ["item_reference"],
                "right_candidates": ["track_identifier"],
            },
        ]
    )

In [77]:
if identifier_key and library_key:
    join_specs.extend(
        [
            {
                "join_name": "identifier_to_library_by_track_id",
                "left_key": identifier_key,
                "right_key": library_key,
                "left_candidates": ["identifier"],
                "right_candidates": ["track_identifier"],
            },
            {
                "join_name": "identifier_to_library_by_catalog_id",
                "left_key": identifier_key,
                "right_key": library_key,
                "left_candidates": ["identifier"],
                "right_candidates": ["apple_music_track_identifier"],
            },
        ]
    )

In [78]:
if imports_key and library_key:
    join_specs.extend(
        [
            {
                "join_name": "imports_to_library_by_imported_id",
                "left_key": imports_key,
                "right_key": library_key,
                "left_candidates": ["imported_library_track_id"],
                "right_candidates": ["track_identifier"],
            },
            {
                "join_name": "imports_to_library_by_matched_ids",
                "left_key": imports_key,
                "right_key": library_key,
                "left_candidates": ["matched_ids"],
                "right_candidates": ["apple_music_track_identifier"],
            },
            {
                "join_name": "imports_to_library_by_track_artist",
                "left_key": imports_key,
                "right_key": library_key,
                "left_candidates": ["track_name", "track_artist_name"],
                "right_candidates": ["title", "artist"],
            },
        ]
    )

In [79]:
if container_key and origin_key:
    join_specs.append(
        {
            "join_name": "container_to_origin_by_description",
            "left_key": container_key,
            "right_key": origin_key,
            "left_candidates": ["container_description"],
            "right_candidates": ["container_description"],
        }
    )

In [80]:
join_rows = []

for spec in join_specs:
    join_rows.append(
        test_join(
            left_key=spec["left_key"],
            right_key=spec["right_key"],
            left_candidates=spec["left_candidates"],
            right_candidates=spec["right_candidates"],
            join_name=spec["join_name"],
        )
    )

In [81]:
joins = pd.DataFrame(join_rows)

if not joins.empty:
    joins = joins.sort_values(
        ["status", "left_overlap_rate", "overlap_unique_values"],
        ascending=[True, False, False],
    ).reset_index(drop=True)

save_table(joins, "join_quality_report.csv")

display(joins)

,join_name,left_table,right_table,left_columns,right_columns,left_unique_values,right_unique_values,overlap_unique_values,left_overlap_rate,right_overlap_rate,status
0,container_to_origin_by_description,raw__csv__container_details,raw__csv__container_origin,container_description,container_description,NaN,NaN,NaN,NaN,NaN,Skipped Missing Column


In [82]:
ready_rows = []

for _, row in contracts.iterrows():
    table_key = row["preferred_key"]
    table_columns = columns.query("table_key == @table_key").copy()

    if table_columns.empty:
        continue

    identifier_count = int((table_columns["column_family"] == "identifier").sum())
    date_count = int((table_columns["column_family"] == "date_time").sum())
    duration_count = int((table_columns["column_family"] == "duration").sum())
    count_count = int((table_columns["column_family"] == "count").sum())
    metadata_count = int((table_columns["column_family"] == "music_metadata").sum())
    context_count = int((table_columns["column_family"] == "context").sum())
    preference_count = int((table_columns["column_family"] == "preference").sum())

    average_missing = table_columns["missing_rate"].mean()
    row_count = int(row["row_count"])
    column_count = int(row["column_count"])

    score = (
        min(row_count / 10_000, 1) * 2.0
        + min(column_count / 40, 1) * 1.0
        + min(identifier_count / 3, 1) * 1.5
        + min(date_count / 2, 1) * 1.0
        + min(duration_count / 2, 1) * 1.0
        + min((metadata_count + context_count + preference_count) / 10, 1) * 1.5
        + (1 - average_missing) * 2.0
    )

    if row["next_use"] == "EDA Core":
        status = "Ready for EDA Core Validation"
    elif row["next_use"] == "EDA Support":
        status = "Ready for EDA Support Validation"
    elif row["next_use"] == "Validation Only":
        status = "Validation Only"
    else:
        status = "Raw Fallback Only"

    ready_rows.append(
        {
            "table_key": table_key,
            "canonical_name": row["canonical_name"],
            "project_role": row["project_role"],
            "next_use": row["next_use"],
            "readiness_status": status,
            "row_count": row_count,
            "column_count": column_count,
            "identifier_columns": identifier_count,
            "date_columns": date_count,
            "duration_columns": duration_count,
            "count_columns": count_count,
            "music_metadata_columns": metadata_count,
            "context_columns": context_count,
            "preference_columns": preference_count,
            "average_missing_rate": round(average_missing, 4),
            "readiness_score": round(score, 3),
        }
    )

In [83]:
readiness = pd.DataFrame(ready_rows)

readiness = readiness.sort_values(
    ["next_use", "readiness_score"],
    ascending=[True, False],
).reset_index(drop=True)

save_table(readiness, "table_readiness.csv")

display(readiness)

,table_key,canonical_name,project_role,next_use,readiness_status,row_count,column_count,identifier_columns,date_columns,duration_columns,count_columns,music_metadata_columns,context_columns,preference_columns,average_missing_rate,readiness_score
0,raw__csv__recently_played_tracks,recently_played_tracks,Recent Track Snapshot,EDA Support,Ready for EDA Support Validation,200,22,2,5,5,3,1,3,0,0.019,6.153
1,raw__csv__recently_played_containers,recently_played_containers,Recent Container Snapshot,EDA Support,Ready for EDA Support Validation,50,11,2,2,0,3,1,2,0,0.226,4.284
2,raw__csv__container_details,container_details,Container Context Table,EDA Support,Ready for EDA Support Validation,162,10,0,2,1,1,3,2,0,0.138,4.257
3,raw__csv__container_origin,container_origin,Container Origin Table,EDA Support,Ready for EDA Support Validation,69,8,0,2,1,1,3,0,0,0.129,3.907
4,raw__csv__recent_impressions,recent_impressions,Recommendation Impression Sample,EDA Support,Ready for EDA Support Validation,359,5,1,1,0,1,0,1,0,0.164,3.019
5,raw__csv__music_play_activity,music_play_activity,Raw Detailed Playback Event Source,Raw Fallback Only,Raw Fallback Only,22872,144,30,8,9,4,5,29,0,0.546,8.909
6,raw__csv__play_history_daily_tracks,play_history_daily_tracks,Raw Summary Listening Source,Raw Fallback Only,Raw Fallback Only,5042,13,2,1,1,3,1,1,0,0.006,5.621
7,raw__csv__feature_statistics,feature_statistics,Validation Summary Table,Validation Only,Validation Only,126,6,0,3,1,0,0,2,0,0.000,3.975
8,raw__csv__top_content,top_content,Validation Summary Table,Validation Only,Validation Only,196,8,0,3,1,2,0,1,0,0.000,3.889
9,raw__csv__play_statistics,play_statistics,Validation Summary Table,Validation Only,Validation Only,36,5,0,3,1,0,0,1,0,0.000,3.782


In [86]:
selected_files = (
    eda_tables
    [
        [
            "canonical_name",
            "preferred_key",
            "project_role",
            "grain",
            "next_use",
            "relative_path",
            "row_count",
            "column_count",
            "reason",
        ]
    ]
    .copy()
    .sort_values(["next_use", "row_count"], ascending=[True, False])
    .reset_index(drop=True)
)

save_table(selected_files, "selected_files_for_eda.csv")

display(selected_files)

,canonical_name,preferred_key,project_role,grain,next_use,relative_path,row_count,column_count,reason
0,recent_impressions,raw__csv__recent_impressions,Recommendation Impression Sample,One recent impression record,EDA Support,data\raw\csv\recent_impressions.csv,359,5,"This table is small and duplicated, so it shou..."
1,recently_played_tracks,raw__csv__recently_played_tracks,Recent Track Snapshot,One recent track summary record,EDA Support,data\raw\csv\recently_played_tracks.csv,200,22,"This table is a recent snapshot, not the main ..."
2,container_details,raw__csv__container_details,Container Context Table,One container detail record,EDA Support,data\raw\csv\container_details.csv,162,10,This may enrich playlist and source analysis b...
3,container_origin,raw__csv__container_origin,Container Origin Table,One container origin summary record,EDA Support,data\raw\csv\container_origin.csv,69,8,This table supports source attribution but is ...
4,recently_played_containers,raw__csv__recently_played_containers,Recent Container Snapshot,One recent container summary record,EDA Support,data\raw\csv\recently_played_containers.csv,50,11,This table is useful for recent context valida...
5,top_content,raw__csv__top_content,Validation Summary Table,One exported top-content summary record,Validation Only,data\raw\csv\top_content.csv,196,8,This is useful for checking computed metrics b...
6,feature_statistics,raw__csv__feature_statistics,Validation Summary Table,One exported feature-statistic summary record,Validation Only,data\raw\csv\feature_statistics.csv,126,6,This is useful for validating source and featu...
7,play_statistics,raw__csv__play_statistics,Validation Summary Table,One exported play-statistic summary record,Validation Only,data\raw\csv\play_statistics.csv,36,5,"This is a validation table, not an event table."


In [88]:
summary = {
    "project_root": str(root),
    "discovered_files": int(
        len(files)
    ),
    "loaded_tables": int(
        len(tables)
    ),
    "failed_tables": int(
        (
            inventory["load_status"]
            == "Failed"
        ).sum()
    ),
    "total_loaded_rows": (
        int(loaded["row_count"].sum())
        if not loaded.empty
        else 0
    ),
    "selected_tables": int(
        len(contracts)
    ),
    "missing_expected_tables": int(
        len(missing_contracts)
    ),
    "tested_joins": int(
        len(joins)
    ),
}

summary_path = (
    tables_root
    / "data_check_summary.json"
)

with open(
    summary_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        summary,
        file,
        indent=4,
        default=str,
    )

print(
    json.dumps(
        summary,
        indent=4,
    )
)

{
    "project_root": "C:\\Users\\sumai\\Documents\\Coding\\my-music-revisits",
    "discovered_files": 25,
    "loaded_tables": 25,
    "failed_tables": 0,
    "total_loaded_rows": 83668,
    "selected_tables": 10,
    "missing_expected_tables": 8,
    "tested_joins": 1
}


In [89]:
print("Tables Directory:", tables_root)
print("EDA Input Table File:", tables_root / "eda_input_tables.csv")
print("Selected Files for EDA File:", tables_root / "selected_files_for_eda.csv")
print("Join Quality Report:", tables_root / "join_quality_report.csv")
print("Table Contracts:", tables_root / "table_contracts.csv")

Tables Directory: C:\Users\sumai\Documents\Coding\my-music-revisits\reports\tables\data_check
EDA Input Table File: C:\Users\sumai\Documents\Coding\my-music-revisits\reports\tables\data_check\eda_input_tables.csv
Selected Files for EDA File: C:\Users\sumai\Documents\Coding\my-music-revisits\reports\tables\data_check\selected_files_for_eda.csv
Join Quality Report: C:\Users\sumai\Documents\Coding\my-music-revisits\reports\tables\data_check\join_quality_report.csv
Table Contracts: C:\Users\sumai\Documents\Coding\my-music-revisits\reports\tables\data_check\table_contracts.csv
